In [8]:
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt

from time import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from collections import OrderedDict

# Load custom modules
from common.functions import *
from common.gradient import *
from common.layers import *
from common.multi_layer_net_extend import MultiLayerNetExtend
from common.multi_layer_net import MultiLayerNet
from common.optimizer import *
from common.trainer import Trainer
from common.util import *

# 한글 폰트 및 마이너스 기호 표시 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [10]:
torch.cuda.init()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.reset_peak_memory_stats(device=None)
print("현재 디바이스:", device)

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

현재 디바이스: cuda


In [11]:
# CIFAR-100 데이터 로드 함수
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

data_path = './cifar-100-python'
# 데이터 경로 설정
train_path = os.path.join(data_path, 'train')
test_path = os.path.join(data_path, 'test')

# 학습 데이터 로드
train_data = unpickle(train_path)
test_data = unpickle(test_path)

# 메타데이터 로드 (클래스 이름 등)
meta_path = os.path.join(data_path, 'meta')
meta_data = unpickle(meta_path)

fine_label_names = [name.decode() for name in meta_data[b'fine_label_names']]
coarse_label_names = [name.decode() for name in meta_data[b'coarse_label_names']]

# 데이터 구조 확인
print("학습 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in train_data.keys()])
print("테스트 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in test_data.keys()])
print("메타 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in meta_data.keys()])

학습 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
테스트 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
메타 데이터 키: ['fine_label_names', 'coarse_label_names']


In [12]:
def one_hot_encode(y, num_classes):
    return np.eye(num_classes)[y.astype(int)]

# train set 으로 validation set 분할
x = train_data[b'data']
x = x.reshape(-1, 3, 32, 32)
y = np.array(train_data[b'coarse_labels'])
z = np.array(train_data[b'fine_labels'])
x_train, x_val, y_train, y_val, z_train, z_val = train_test_split(x, y, z, test_size=0.2, random_state=42, stratify=y)

# test set 정의
x_test = test_data[b'data']
x_test = x_test.reshape(-1, 3, 32, 32)
y_test = np.array(test_data[b'coarse_labels'])
z_test = np.array(test_data[b'fine_labels'])

x_test = x_test.astype(np.float32) / 255.0
x_train = x_train.astype(np.float32) / 255.0
x_val = x_val.astype(np.float32) / 255.0

y_train = one_hot_encode(y_train, 20)
y_val = one_hot_encode(y_val, 20)
y_test = one_hot_encode(y_test, 20)

z_train = one_hot_encode(z_train, 100)
z_val = one_hot_encode(z_val, 100)
z_test = one_hot_encode(z_test, 100)

x_train.shape, x_val.shape, x_test.shape, y_train.shape, y_val.shape, y_test.shape, z_train.shape, z_val.shape, z_test.shape

((40000, 3, 32, 32),
 (10000, 3, 32, 32),
 (10000, 3, 32, 32),
 (40000, 20),
 (10000, 20),
 (10000, 20),
 (40000, 100),
 (10000, 100),
 (10000, 100))

In [20]:
class CIFAR100Dataset(Dataset):
    def __init__(self, images, labels_coarse, labels_fine, transform=None): # transform 인자 추가
        self.images = images
        self.labels_coarse = labels_coarse
        self.labels_fine = labels_fine
        self.transform = transform # transform 저장

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx] # (C, H, W) 형태의 NumPy 배열
        label_coarse = self.labels_coarse[idx]
        label_fine = self.labels_fine[idx]
        
        # NumPy 배열을 PyTorch 텐서로 변환
        # 이미지는 이미 float32로 정규화되어 있음
        image_tensor = torch.tensor(image) 
        label_coarse_tensor = torch.tensor(label_coarse) # 레이블은 원-핫 인코딩된 상태
        label_fine_tensor = torch.tensor(label_fine) # 레이블은 원-핫 인코딩된 상태
        
        if self.transform:
            image_tensor = self.transform(image_tensor) # 변환 적용
            
        return image_tensor, label_coarse_tensor, label_fine_tensor

In [21]:
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # 이미지 주위에 4픽셀 패딩 후 32x32 랜덤 크롭
    transforms.RandomHorizontalFlip(p=0.5), # 50% 확률로 좌우 반전
    # 필요한 경우 다른 변환 추가 가능 (예: transforms.ColorJitter)
])
train_dataset = CIFAR100Dataset(x_train, y_train, z_train, transform=train_transforms)
val_dataset = CIFAR100Dataset(x_val, y_val, z_val)
test_dataset = CIFAR100Dataset(x_test, y_test, z_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [22]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Conv1: Input (3, 32, 32) -> Output (32, 16, 16)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1, stride=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv2: Input (32, 16, 16) -> Output (64, 8, 8)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, stride=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv3: Input (64, 8, 8) -> Output (128, 8, 8)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1, stride=1)
        self.relu3 = nn.ReLU()
        # No pooling after conv3

        # Calculate the flattened size after conv3
        # Output of conv3 will be (batch_size, 128, 8, 8)
        self.flattened_size = 128 * 8 * 8

        # Affine1 (FC1)
        self.fc1 = nn.Linear(self.flattened_size, 128)
        self.relu4 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5) # 과적합 방지를 위해 드롭아웃 추가 고려

        # Affine2 (FC2 - Output layer)
        self.fc2 = nn.Linear(128, 20) # 20 coarse labels

    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Conv block 2
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # Conv block 3
        x = self.conv3(x)
        x = self.relu3(x)

        # Flatten
        x = x.view(-1, self.flattened_size)

        # FC block 1
        x = self.fc1(x)
        x = self.relu4(x)
        x = self.dropout1(x) # 드롭아웃 적용
        
        # FC block 2 (Output)
        x = self.fc2(x)
        return x

In [23]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Convolutional Block 1 (Inspired by Keras example)
        self.conv1_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1) # padding='same'
        self.relu1_1 = nn.ReLU()
        self.conv1_2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) # padding='same'
        self.relu1_2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # (32, 16, 16)

        # Convolutional Block 2 (Inspired by Keras example)
        self.conv2_1 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) # padding='same'
        self.relu2_1 = nn.ReLU()
        self.conv2_2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1) # padding='same'
        self.relu2_2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # (128, 8, 8)
        
        self.flattened_size = 128 * 8 * 8 # 8192
        
        # Shared Dense Block (before branching)
        self.fc_shared = nn.Linear(self.flattened_size, 256)
        self.relu_fc_shared = nn.ReLU()
        self.bn_fc_shared = nn.BatchNorm1d(256)
        self.drop_fc_shared = nn.Dropout(p=0.3)

        # --- Branch for Coarse Labels (20 classes) ---
        self.fc_coarse_1 = nn.Linear(256, 256)
        self.relu_fc_coarse_1 = nn.ReLU()
        self.bn_fc_coarse_1 = nn.BatchNorm1d(256)
        self.drop_fc_coarse_1 = nn.Dropout(p=0.3)
        self.fc_out_coarse = nn.Linear(256, 20) # Output for 20 coarse labels

        # --- Branch for Fine Labels (100 classes) ---
        self.fc_fine_1 = nn.Linear(256, 512) # Potentially more capacity for fine labels
        self.relu_fc_fine_1 = nn.ReLU()
        self.bn_fc_fine_1 = nn.BatchNorm1d(512)
        self.drop_fc_fine_1 = nn.Dropout(p=0.4) # Potentially different dropout

        self.fc_fine_2 = nn.Linear(512, 256)
        self.relu_fc_fine_2 = nn.ReLU()
        self.bn_fc_fine_2 = nn.BatchNorm1d(256)
        self.drop_fc_fine_2 = nn.Dropout(p=0.4)
        self.fc_out_fine = nn.Linear(256, 100) # Output for 100 fine labels

    def forward(self, x):
        # Conv Block 1
        x = self.relu1_1(self.conv1_1(x))
        x = self.relu1_2(self.conv1_2(x))
        x = self.pool1(x)
        
        # Conv Block 2
        x = self.relu2_1(self.conv2_1(x))
        x = self.relu2_2(self.conv2_2(x))
        x = self.pool2(x)
        
        # Flatten
        x = x.view(-1, self.flattened_size)
        
        # Shared Dense Block
        shared_features = self.fc_shared(x)
        shared_features = self.relu_fc_shared(shared_features)
        shared_features = self.bn_fc_shared(shared_features)
        shared_features = self.drop_fc_shared(shared_features)
        
        # Coarse Label Branch
        x_coarse = self.fc_coarse_1(shared_features)
        x_coarse = self.relu_fc_coarse_1(x_coarse)
        x_coarse = self.bn_fc_coarse_1(x_coarse)
        x_coarse = self.drop_fc_coarse_1(x_coarse)
        out_coarse = self.fc_out_coarse(x_coarse)
        
        # Fine Label Branch
        x_fine = self.fc_fine_1(shared_features) # Use shared_features as input
        x_fine = self.relu_fc_fine_1(x_fine)
        x_fine = self.bn_fc_fine_1(x_fine)
        x_fine = self.drop_fc_fine_1(x_fine)

        x_fine = self.fc_fine_2(x_fine)
        x_fine = self.relu_fc_fine_2(x_fine)
        x_fine = self.bn_fc_fine_2(x_fine)
        x_fine = self.drop_fc_fine_2(x_fine)
        out_fine = self.fc_out_fine(x_fine)
        
        return out_coarse, out_fine

In [36]:
model = SimpleCNN().to(device)
criterion_coarse = nn.CrossEntropyLoss()
criterion_fine = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=50, gamma=0.5)  # 10 에폭마다 학습률을 0.1배로 감소

In [37]:
num_epochs = 200
# coarse와 fine 레이블의 가중치 설정
lambda_coarse = 0.4
lambda_fine = 0.6

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train_coarse = 0
    correct_train_fine = 0
    total_train = 0
    for i, (images, labels_coarse, labels_fine) in enumerate(train_loader):
        images = images.to(device)
        labels_coarse = labels_coarse.to(device) # (batch_size, 20)
        labels_fine = labels_fine.to(device)     # (batch_size, 100)

        # Convert one-hot to class indices for CrossEntropyLoss
        target_labels_coarse = torch.argmax(labels_coarse, dim=1) # (batch_size)
        target_labels_fine = torch.argmax(labels_fine, dim=1)     # (batch_size)

        optimizer.zero_grad()
        out_coarse, out_fine = model(images)

        loss_c = criterion_coarse(out_coarse, target_labels_coarse)
        loss_f = criterion_fine(out_fine, target_labels_fine)
        total_loss = lambda_coarse * loss_c + lambda_fine * loss_f # Weighted sum

        total_loss.backward()
        optimizer.step()
        
        running_loss += total_loss.item() * images.size(0)
        
        _, predicted_coarse_train = torch.max(out_coarse.data, 1)
        _, predicted_fine_train = torch.max(out_fine.data, 1)
        
        total_train += images.size(0) # labels_coarse.size(0) or labels_fine.size(0)
        correct_train_coarse += (predicted_coarse_train == target_labels_coarse).sum().item()
        correct_train_fine += (predicted_fine_train == target_labels_fine).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc_coarse = 100 * correct_train_coarse / total_train
    epoch_train_acc_fine = 100 * correct_train_fine / total_train
    
    # --- 검증 단계 ---
    model.eval()
    val_loss_coarse_total = 0.0
    val_loss_fine_total = 0.0
    val_loss_combined_total = 0.0
    correct_val_coarse = 0
    correct_val_fine = 0
    total_val = 0
    with torch.no_grad():
        for images, labels_coarse, labels_fine in val_loader:
            images = images.to(device)
            labels_coarse = labels_coarse.to(device)
            labels_fine = labels_fine.to(device)

            target_labels_coarse = torch.argmax(labels_coarse, dim=1)
            target_labels_fine = torch.argmax(labels_fine, dim=1)
            
            out_coarse, out_fine = model(images)
            
            loss_c = criterion_coarse(out_coarse, target_labels_coarse)
            loss_f = criterion_fine(out_fine, target_labels_fine)
            combined_loss = lambda_coarse * loss_c + lambda_fine * loss_f

            val_loss_coarse_total += loss_c.item() * images.size(0)
            val_loss_fine_total += loss_f.item() * images.size(0)
            val_loss_combined_total += combined_loss.item() * images.size(0)
            
            _, predicted_val_coarse = torch.max(out_coarse.data, 1)
            _, predicted_val_fine = torch.max(out_fine.data, 1)
            
            total_val += images.size(0)
            correct_val_coarse += (predicted_val_coarse == target_labels_coarse).sum().item()
            correct_val_fine += (predicted_val_fine == target_labels_fine).sum().item()
            
    epoch_val_loss_combined = val_loss_combined_total / total_val
    epoch_val_acc_coarse = 100 * correct_val_coarse / total_val
    epoch_val_acc_fine = 100 * correct_val_fine / total_val
    
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, Train Coarse Acc: {epoch_train_acc_coarse:.2f}%, Train Fine Acc: {epoch_train_acc_fine:.2f}%, '
          f'Val Loss: {epoch_val_loss_combined:.4f}, Val Coarse Acc: {epoch_val_acc_coarse:.2f}%, Val Fine Acc: {epoch_val_acc_fine:.2f}%')
    scheduler.step()  # 학습률 스케줄러 업데이트

# --- 최종 테스트 단계 ---
model.eval()
correct_test_coarse = 0
correct_test_fine = 0
total_test = 0
test_loss_coarse_total = 0.0
test_loss_fine_total = 0.0
test_loss_combined_total = 0.0

with torch.no_grad():
    for images, labels_coarse, labels_fine in test_loader: # test_loader 사용
        images = images.to(device)
        labels_coarse = labels_coarse.to(device)
        labels_fine = labels_fine.to(device)

        target_labels_coarse = torch.argmax(labels_coarse, dim=1)
        target_labels_fine = torch.argmax(labels_fine, dim=1)
        
        out_coarse, out_fine = model(images)

        loss_c = criterion_coarse(out_coarse, target_labels_coarse)
        loss_f = criterion_fine(out_fine, target_labels_fine)
        combined_loss = lambda_coarse * loss_c + lambda_fine * loss_f

        test_loss_coarse_total += loss_c.item() * images.size(0)
        test_loss_fine_total += loss_f.item() * images.size(0)
        test_loss_combined_total += combined_loss.item() * images.size(0)

        _, predicted_test_coarse = torch.max(out_coarse.data, 1)
        _, predicted_test_fine = torch.max(out_fine.data, 1)

        total_test += images.size(0)
        correct_test_coarse += (predicted_test_coarse == target_labels_coarse).sum().item()
        correct_test_fine += (predicted_test_fine == target_labels_fine).sum().item()

final_test_loss_combined = test_loss_combined_total / total_test
final_test_acc_coarse = 100 * correct_test_coarse / total_test
final_test_acc_fine = 100 * correct_test_fine / total_test

print(f'Test Loss (Combined): {final_test_loss_combined:.4f}')
print(f'Test Coarse Accuracy on {total_test} test images: {final_test_acc_coarse:.2f}%')
print(f'Test Fine Accuracy on {total_test} test images: {final_test_acc_fine:.2f}%')

Epoch [1/200], Train Loss: 3.6588, Train Coarse Acc: 16.00%, Train Fine Acc: 5.43%, Val Loss: 3.7199, Val Coarse Acc: 16.61%, Val Fine Acc: 6.42%
Epoch [2/200], Train Loss: 3.3167, Train Coarse Acc: 22.42%, Train Fine Acc: 9.55%, Val Loss: 3.2580, Val Coarse Acc: 25.68%, Val Fine Acc: 10.93%
Epoch [3/200], Train Loss: 3.1827, Train Coarse Acc: 25.36%, Train Fine Acc: 11.89%, Val Loss: 4.0867, Val Coarse Acc: 28.84%, Val Fine Acc: 14.87%
Epoch [4/200], Train Loss: 3.0748, Train Coarse Acc: 28.12%, Train Fine Acc: 14.05%, Val Loss: 3.4310, Val Coarse Acc: 29.82%, Val Fine Acc: 16.91%
Epoch [5/200], Train Loss: 2.9859, Train Coarse Acc: 29.99%, Train Fine Acc: 15.66%, Val Loss: 2.9662, Val Coarse Acc: 31.15%, Val Fine Acc: 16.64%
Epoch [6/200], Train Loss: 2.9113, Train Coarse Acc: 31.49%, Train Fine Acc: 17.00%, Val Loss: 2.7821, Val Coarse Acc: 35.55%, Val Fine Acc: 19.81%
Epoch [7/200], Train Loss: 2.8549, Train Coarse Acc: 33.10%, Train Fine Acc: 18.36%, Val Loss: 3.8632, Val Coarse A